<a href="https://colab.research.google.com/github/lricci03/Hands-on-ML/blob/main/c13_ex10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 13 - Exercise 13
The Bach chorales dataset.
Composed of 382 chorales. Each chorale is 100 to 640 time steps long, and each time step contains 4 integers, where each integer corresponds to a note's index on a piano (except for the value 0, which means that no note is played).

Train a model - recurrent, convolutional or both - that can predict the next time step (four notes), given a sequence of time steps from a chorale.

Then use this model to generate Bach-like music, one note at a time:
you can do this by giving the model the start of a chorale and asking it to predict the next time step, then appending these time steps to the input sequence and asking the model for the next note, and so on.

Google's Coconet model was used for a nice Google doodle about Bach.

In [2]:
import torch
import torch.nn as nn

if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'


Connect to Google drive

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [6]:

from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales"
)

print(DATA_DIR.exists())

True


## Import the data

The directory jsb_chorales contains three directories: train, test, valid. Each contains csv files *chorale_i.csv* to *chorale_j.csv*. Each file *chorale_i.csv* has four columns: note0, note1, note2, note3 and between 100 and 640 rows.

* train: 000 to 228
* valid: 229 to 304
* test: 305 to 381


In [7]:
import pandas as pd

Example of one chorale

In [8]:
path = Path("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/train/chorale_000.csv")
df = pd.read_csv(path)
df.head()


,note0,note1,note2,note3
0,74,70,65,58
1,74,70,65,58
2,74,70,65,58
3,74,70,65,58
4,75,70,58,55


In [12]:
chorale_000 = torch.Tensor(df[['note0','note1','note2','note3']].values)

## Prepare the data: Multivariate time series dataset
Each time step has four notes, so we have four variables.

For a given time series `series`, we create sliding windows of the given `window_length`.

The target is one time step ahead of the window, and also contains four variables.

In [33]:
class MulvarTimeSeriesDataset(torch.utils.data.Dataset):
  def __init__(self, series, window_length):
    self.series = series
    self.window_length = window_length

  def __len__(self):
    return len(self.series) - self.window_length

  def __getitem__(self,idx):
    if idx >= len(self):
      raise IndexError('dataset index out of range')
    end = idx + self.window_length
    window = self.series[idx : end]
    target = self.series[end]
    return window, target

We want to slice each each chorale_xxx.csv in time windows and then use all these windows as dataset.

If we concatenate all csv's into a single one, we will get time windows that bridge over different chorales, which we don't want.

Therefore we import each chorale_xxx.csv as pandas dataframe, convert it into tensor, slice it into windows and add it to a list `datasets`.

We then concatenate the list into a single dataset using `ConcatDataset` and finally feed the dataset to `DataLoader`.

Create a list of paths to all the files in the directories train, valid, test.

`os.listdir()` creates a list of all the names of files in a given directory.

In [18]:
import os

train_list = os.listdir("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/train")

print(train_list[0])
print("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/train/"+train_list[0])

chorale_000.csv
/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/train/chorale_000.csv


In [21]:
import os

train_list = os.listdir("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/train")
valid_list = os.listdir("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/valid")
test_list = os.listdir("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/test")

train_paths = []
valid_paths = []
test_paths = []

for name in train_list:
  train_paths.append("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/train/"+name)
for name in valid_list:
  valid_paths.append("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/valid/"+name)
for name in test_list:
  test_paths.append("/content/drive/MyDrive/Hands_on_ML/c13/jsb_chorales/test/"+name)

Create train dataset and dataloader

In [34]:
from torch.utils.data import ConcatDataset, DataLoader

window_length = 40
datasets = []
for path in train_paths:
  df = pd.read_csv(path)
  chorale = torch.Tensor(df[['note0','note1','note2','note3']].values)
  chorale_windows = MulvarTimeSeriesDataset(chorale, window_length)
  datasets.append(chorale_windows)

train_set = ConcatDataset(datasets)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)

Create valid and test dataset and dataloader

In [35]:
window_length = 40
datasets = []
for path in valid_paths:
  df = pd.read_csv(path)
  chorale = torch.Tensor(df[['note0','note1','note2','note3']].values)
  chorale_windows = MulvarTimeSeriesDataset(chorale, window_length)
  datasets.append(chorale_windows)

valid_set = ConcatDataset(datasets)
valid_loader = DataLoader(valid_set, batch_size=32)

datasets = []
for path in test_paths:
  df = pd.read_csv(path)
  chorale = torch.Tensor(df[['note0','note1','note2','note3']].values)
  chorale_windows = MulvarTimeSeriesDataset(chorale, window_length)
  datasets.append(chorale_windows)

test_set = ConcatDataset(datasets)
test_loader = DataLoader(test_set, batch_size=32)

## Forecast using a simple RNN model

In [27]:
class SimpleRnnModel(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
    self.output = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    outputs, last_state = self.rnn(X)
    return self.output(outputs[:, -1])

## train and evaluation functions

In [28]:
%pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 57.0 MB/s eta 0:00:00


**NOTE** that we round the predictions before evaluating them.

**ABSOLUTELY** don't round during training, otherwise the loss function will have zero gradients.

In [55]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            # round the predictions to integers
            y_pred_rounded = torch.round(y_pred)
            metric.update(y_pred_rounded, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=20, factor=0.1):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # rounding prediction for train metric
            # .detach() ensures that no unintented history is tracked for the metric
            y_pred_rounded = torch.round(y_pred.detach())
            metric.update(y_pred_rounded, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [56]:
torch.manual_seed(42)

simple_model = SimpleRnnModel(input_size=4, hidden_size=32, output_size=4).to(device)

In [60]:
loss_fn = nn.HuberLoss()
optimizer = torch.optim.SGD(simple_model.parameters(), lr=0.5, momentum=0.95)
metric = torchmetrics.MeanAbsoluteError().to(device)

history = train(simple_model, optimizer, loss_fn, metric, train_loader,
                valid_loader, n_epochs=50, patience=5, factor=0.5)

Epoch 1/50, train loss: 3.2616, train metric: 3.7263, valid metric: 3.4425
Epoch 2/50, train loss: 3.2173, train metric: 3.6770, valid metric: 3.2700
Epoch 3/50, train loss: 3.2193, train metric: 3.6858, valid metric: 3.3282
Epoch 4/50, train loss: 3.0848, train metric: 3.5475, valid metric: 3.5073
Epoch 5/50, train loss: 3.2246, train metric: 3.6884, valid metric: 3.8028
Epoch 6/50, train loss: 3.1257, train metric: 3.5884, valid metric: 3.8079
Epoch 7/50, train loss: 3.1973, train metric: 3.6594, valid metric: 3.8203
Epoch 8/50, train loss: 3.1870, train metric: 3.6535, valid metric: 3.6309
Epoch 9/50, train loss: 2.8655, train metric: 3.3247, valid metric: 3.4245
Epoch 10/50, train loss: 2.8809, train metric: 3.3425, valid metric: 3.2025
Epoch 11/50, train loss: 2.9304, train metric: 3.3903, valid metric: 3.5932
Epoch 12/50, train loss: 2.8893, train metric: 3.3523, valid metric: 3.1556
Epoch 13/50, train loss: 2.9060, train metric: 3.3649, valid metric: 3.3040
Epoch 14/50, train lo

## Analyze predictions

In [38]:
X = valid_set

In [50]:
X[0]

(tensor([[72., 67., 60., 48.],
         [72., 67., 60., 48.],
         [72., 67., 60., 48.],
         [72., 67., 60., 48.],
         [72., 67., 64., 48.],
         [72., 67., 64., 48.],
         [72., 67., 64., 50.],
         [72., 67., 64., 50.],
         [72., 67., 64., 52.],
         [72., 67., 64., 52.],
         [72., 67., 64., 53.],
         [72., 67., 64., 53.],
         [71., 67., 62., 55.],
         [71., 67., 62., 55.],
         [71., 65., 62., 55.],
         [71., 65., 62., 55.],
         [69., 64., 60., 45.],
         [69., 64., 60., 45.],
         [69., 64., 60., 47.],
         [69., 64., 60., 47.],
         [69., 64., 60., 48.],
         [69., 64., 60., 48.],
         [69., 65., 60., 50.],
         [69., 65., 60., 50.],
         [67., 67., 60., 52.],
         [67., 67., 60., 52.],
         [67., 67., 60., 52.],
         [67., 67., 60., 52.],
         [72., 64., 60., 57.],
         [72., 64., 60., 57.],
         [72., 64., 60., 57.],
         [72., 64., 60., 57.],
        

In [48]:
X[0][0].shape

torch.Size([40, 4])

In [49]:
X[0][0].unsqueeze(dim=0).shape

torch.Size([1, 40, 4])

In [47]:
X[0][1]

tensor([76., 67., 64., 48.])

valid_set is a dataset in which each element valid_set[i] is a tuple of two tensors: (window, target) (cfr. [how we prepared the data](#scrollTo=yPyCk_Ard5EH&line=8&uniqifier=1))

The tensor window is a list of 4-element lists and has shape [40,4], the tensor target is a 4-element list and has shape [4]

valid_set[0] consists of the first 40 timesteps of the first csv in the directory valid, that is, chorale_229.csv (window) and the 41th timesteps (target).

valid_set[1] is the shift by 1: timesteps 1-41 plus 42th timestep as target.

Let's predict `valid_set[0][0]` using our simple RNN model and compare it with the actual target `valid_set[0][1]`

In [61]:
simple_model.eval()
with torch.no_grad():
  X = valid_set[0][0].unsqueeze(dim=0)
  Y_pred = simple_model(X.to(device))
print(Y_pred)
print(valid_set[0][1])

tensor([[70.6301, 65.6136, 59.5504, 51.0594]], device='cuda:0')
tensor([76., 67., 64., 48.])


Does it do better on the train set?

In [62]:
simple_model.eval()
with torch.no_grad():
  X = train_set[0][0].unsqueeze(dim=0)
  Y_pred = simple_model(X.to(device))
print(Y_pred)
print(train_set[0][1])

tensor([[70.6301, 65.6136, 59.5504, 51.0594]], device='cuda:0')
tensor([77., 65., 62., 50.])


## Forecast using a deep RNN

In [63]:
class DeepRnnModel(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.deeprnn = nn.RNN(input_size, hidden_size, num_layers=3, batch_first=True)
    self.output = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    outputs, last_state = self.deeprnn(X)
    return self.output(outputs[:, -1])

In [67]:
torch.manual_seed(42)

deep_model = DeepRnnModel(input_size=4, hidden_size=32, output_size=4).to(device)

In [69]:
loss_fn = nn.HuberLoss()
optimizer = torch.optim.SGD(deep_model.parameters(), lr=0.1, momentum=0.95)
metric = torchmetrics.MeanAbsoluteError().to(device)

history = train(deep_model, optimizer, loss_fn, metric, train_loader,
                valid_loader, n_epochs=50, patience=5, factor=0.5)

Epoch 1/50, train loss: 2.7688, train metric: 3.2281, valid metric: 3.5354
Epoch 2/50, train loss: 2.7845, train metric: 3.2419, valid metric: 3.4949
Epoch 3/50, train loss: 2.7565, train metric: 3.2126, valid metric: 3.4362
Epoch 4/50, train loss: 2.7723, train metric: 3.2286, valid metric: 3.8180
Epoch 5/50, train loss: 2.7735, train metric: 3.2313, valid metric: 3.4978
Epoch 6/50, train loss: 2.7863, train metric: 3.2451, valid metric: 3.5603
Epoch 7/50, train loss: 2.7834, train metric: 3.2405, valid metric: 3.5135
Epoch 8/50, train loss: 2.7697, train metric: 3.2258, valid metric: 3.4982
Epoch 9/50, train loss: 2.7744, train metric: 3.2312, valid metric: 3.7212
Epoch 10/50, train loss: 2.7283, train metric: 3.1862, valid metric: 3.4219
Epoch 11/50, train loss: 2.7409, train metric: 3.1979, valid metric: 3.4128
Epoch 12/50, train loss: 2.7132, train metric: 3.1658, valid metric: 3.5201
Epoch 13/50, train loss: 2.7121, train metric: 3.1681, valid metric: 3.3986
Epoch 14/50, train lo

Deep RNN performs worse than simple RNN

In [75]:
deep_model.eval()
with torch.no_grad():
  X = valid_set[0][0].unsqueeze(dim=0)
  Y_pred = deep_model(X.to(device))
print(Y_pred)
print(valid_set[0][1])

tensor([[70.5130, 65.0189, 59.4020, 50.7668]], device='cuda:0')
tensor([76., 67., 64., 48.])


In [89]:
len(valid_set)

15368

In [90]:
deep_model.eval()
with torch.no_grad():
  X = valid_set[10000][0].unsqueeze(dim=0)
  Y_pred = deep_model(X.to(device))
print(Y_pred)
print(valid_set[10000][1])

tensor([[70.5130, 65.0189, 59.4020, 50.7668]], device='cuda:0')
tensor([71., 62., 59., 47.])


## Generate Bach-Like music

We generate a chorale 100 timesteps long starting from the first chorale of the validation set

In [91]:
n_steps = 100
simple_model.eval()
with torch.no_grad():
  X = valid_set[10000][0].unsqueeze(dim=0).to(device)
  for step_ahead in range(n_steps):
    y_pred_one = simple_model(X)
    y_pred_rounded = torch.round(y_pred_one)
    X = torch.cat([X, y_pred_rounded.unsqueeze(dim=1)], dim=1) # X has shape [1,40,4] and y_pred_one has shape [1,4]. .unsqueeze(dim=1) adds a dimension in the middle. We concatenate along that.

  new_chorale = X[0, -n_steps:, :]

In [92]:
new_chorale

tensor([[71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [7

## Increase time window

The predictions of consecutive 100 time steps are constant. Let's try to increase the time window of our data set.

In [93]:
window_length = 60
datasets = []
for path in train_paths:
  df = pd.read_csv(path)
  chorale = torch.Tensor(df[['note0','note1','note2','note3']].values)
  chorale_windows = MulvarTimeSeriesDataset(chorale, window_length)
  datasets.append(chorale_windows)

longer_train_set = ConcatDataset(datasets)
longer_train_loader = DataLoader(longer_train_set, batch_size=32, shuffle=True)

In [94]:
datasets = []
for path in valid_paths:
  df = pd.read_csv(path)
  chorale = torch.Tensor(df[['note0','note1','note2','note3']].values)
  chorale_windows = MulvarTimeSeriesDataset(chorale, window_length)
  datasets.append(chorale_windows)

longer_valid_set = ConcatDataset(datasets)
longer_valid_loader = DataLoader(longer_valid_set, batch_size=32)

datasets = []
for path in test_paths:
  df = pd.read_csv(path)
  chorale = torch.Tensor(df[['note0','note1','note2','note3']].values)
  chorale_windows = MulvarTimeSeriesDataset(chorale, window_length)
  datasets.append(chorale_windows)

longer_test_set = ConcatDataset(datasets)
longer_test_loader = DataLoader(longer_test_set, batch_size=32)

In [96]:
torch.manual_seed(42)

longer_simple_model = SimpleRnnModel(input_size=4, hidden_size=32, output_size=4).to(device)

In [97]:
loss_fn = nn.HuberLoss()
optimizer = torch.optim.SGD(longer_simple_model.parameters(), lr=0.5, momentum=0.95)
metric = torchmetrics.MeanAbsoluteError().to(device)

history = train(longer_simple_model, optimizer, loss_fn, metric, longer_train_loader,
                longer_valid_loader, n_epochs=50, patience=5, factor=0.5)

Epoch 1/50, train loss: 3.8290, train metric: 4.2942, valid metric: 4.4535
Epoch 2/50, train loss: 3.2739, train metric: 3.7383, valid metric: 3.3135
Epoch 3/50, train loss: 3.1813, train metric: 3.6409, valid metric: 4.2458
Epoch 4/50, train loss: 3.1611, train metric: 3.6231, valid metric: 3.5148
Epoch 5/50, train loss: 3.2093, train metric: 3.6719, valid metric: 3.4584
Epoch 6/50, train loss: 3.2094, train metric: 3.6702, valid metric: 3.2741
Epoch 7/50, train loss: 3.1973, train metric: 3.6598, valid metric: 3.6426
Epoch 8/50, train loss: 3.2505, train metric: 3.7151, valid metric: 4.2785
Epoch 9/50, train loss: 3.1921, train metric: 3.6550, valid metric: 4.6657
Epoch 10/50, train loss: 3.2329, train metric: 3.6919, valid metric: 4.0920
Epoch 11/50, train loss: 3.2653, train metric: 3.7293, valid metric: 3.9980
Epoch 12/50, train loss: 3.2290, train metric: 3.6940, valid metric: 3.8004
Epoch 13/50, train loss: 2.9294, train metric: 3.3911, valid metric: 3.3552
Epoch 14/50, train lo

In [99]:
longer_simple_model.eval()
with torch.no_grad():
  X = valid_set[0][0].unsqueeze(dim=0)
  Y_pred = longer_simple_model(X.to(device))
print(Y_pred)
print(valid_set[0][1])

tensor([[71.2987, 65.5399, 59.7374, 51.3451]], device='cuda:0')
tensor([76., 67., 64., 48.])


In [100]:
n_steps = 100
longer_simple_model.eval()
with torch.no_grad():
  X = valid_set[10000][0].unsqueeze(dim=0).to(device)
  for step_ahead in range(n_steps):
    y_pred_one = longer_simple_model(X)
    y_pred_rounded = torch.round(y_pred_one)
    X = torch.cat([X, y_pred_rounded.unsqueeze(dim=1)], dim=1) # X has shape [1,40,4] and y_pred_one has shape [1,4]. .unsqueeze(dim=1) adds a dimension in the middle. We concatenate along that.

  chorale_long_model = X[0, -n_steps:, :]

In [101]:
chorale_long_model

tensor([[71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [71., 66., 60., 51.],
        [7

## LSTM Model

In [ ]:
class LstmModel(nn.Module):